In [ ]:
import pandas as pd
from pathlib import Path
from typing import List, Optional
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from torch import nn
import torch
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:

def merge_csv(
    folder_path: Optional[Path] = None, 
    file_paths: Optional[List[Path]] = None
) -> pd.DataFrame:
    """
    Merges CSV files from either a specified folder or a list of file paths.

    Args:
        folder_path: The path to the folder containing CSV files to merge.
        file_paths: A list of specific CSV file paths to merge.

    Returns:
        A single pandas DataFrame containing the merged data.

    Raises:
        ValueError: If neither or both `folder_path` and `file_paths` are provided,
                    or if no CSV files are found.
    """
    # 1. Input Validation (Guard Clauses)
    # This ensures the function is called correctly.
    if (folder_path is None and file_paths is None) or \
       (folder_path is not None and file_paths is not None):
        raise ValueError("Provide either 'folder_path' OR 'file_paths', not both/neither.")

    # 2. Determine the list of files based on the mode
    if folder_path:
        files = list(folder_path.glob("*.csv"))
    else:  # This block runs if file_paths is provided
        files = file_paths

    if not files:
        raise ValueError("No CSV files found to merge.")

    # 3. Efficient Merging Logic
    # Reading all files into a list of DataFrames and concatenating once is
    # far more performant than concatenating one-by-one in a loop.
    df_list = (pd.read_csv(file) for file in files)
    return pd.concat(df_list, ignore_index=True)

In [ ]:
from scipy.signal import detrend

TRAIN_DATA_PATH = Path("small_train_data")
VALID_DATA_PATH = Path("small_valid_data")

# Load TRAINING data
print("=== Loading Training Data ===")
df_train = merge_csv(folder_path=TRAIN_DATA_PATH)
print(f"Training data shape: {df_train.shape}")
print(f"Training labels: {df_train['label'].value_counts().to_dict()}")

# Load VALIDATION data  
print("\n=== Loading Validation Data ===")
df_valid = merge_csv(folder_path=VALID_DATA_PATH)
print(f"Validation data shape: {df_valid.shape}")
print(f"Validation labels: {df_valid['label'].value_counts().to_dict()}")

print(f"\nAvailable columns: {df_train.columns.tolist()[:10]}...")

In [ ]:
# === Temporal Feature Extraction from Packet-Level WiFi Data ===

def extract_temporal_features(df):
    """
    Extract temporal derivative and motion-related features from packet-level WiFi data.
    
    IMPORTANT: This data contains packet-level features (pkt0-pkt59), NOT raw CSI.
    Therefore, we extract temporal derivatives as motion proxies rather than 
    true Doppler velocities (which require CSI phase information).
    
    Features extracted:
    1. Rate of change (1st derivative) - detects motion presence
    2. Acceleration (2nd derivative) - measures activity intensity
    3. Jerk (3rd derivative) - captures abrupt motion changes
    4. Multi-scale variance - motion consistency over time
    5. Signal energy - accumulated motion magnitude
    6. Statistical aggregates - activity signatures
    """
    print(f"Input columns: {df.columns.tolist()[:10]}... ({len(df.columns)} total)")
    
    temporal_features = pd.DataFrame(index=df.index)
    
    # Get all numeric columns except label
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if 'label' in numeric_cols:
        numeric_cols.remove('label')
    
    if len(numeric_cols) == 0:
        print("Warning: No numeric columns found for feature extraction")
        return df
    
    # === 1. First-Order Temporal Derivatives (Rate of Change) ===
    # Captures: How fast packet features are changing → motion proxy
    # High values indicate movement, low values indicate static state
    for col in numeric_cols[:15]:
        rate_of_change = df[col].diff().fillna(0)
        temporal_features[f'rate_of_change_{col}'] = rate_of_change.replace([np.inf, -np.inf], 0)
    
    # === 2. Second-Order Derivatives (Acceleration Proxy) ===
    # Captures: How the rate of change is changing → activity intensity
    # Distinguishes: walking (periodic) vs sitting (constant) vs falling (spike)
    for col in numeric_cols[:10]:
        acceleration_proxy = df[col].diff().diff().fillna(0)
        temporal_features[f'acceleration_{col}'] = acceleration_proxy.replace([np.inf, -np.inf], 0)
    
    # === 3. Third-Order Derivatives (Jerk - Abrupt Changes) ===
    # Captures: Sudden motion transitions
    # High jerk: falling, standing up, sitting down
    # Low jerk: steady walking, sleeping, sitting
    for col in numeric_cols[:5]:
        jerk = df[col].diff().diff().diff().fillna(0)
        temporal_features[f'jerk_{col}'] = jerk.replace([np.inf, -np.inf], 0)
    
    # === 4. Multi-Scale Variance Features (Motion Patterns) ===
    # Different time windows capture different motion characteristics:
    # - Short window (5): rapid movements (e.g., hand gestures)
    # - Medium window (10): periodic patterns (e.g., walking steps)
    # - Long window (20): overall activity level
    windows = [5, 10, 20]
    for window in windows:
        window_size = min(window, max(2, len(df) // 100))
        
        for col in numeric_cols[:8]:
            # Variance of rate of change (motion consistency)
            rate_change = df[col].diff()
            roll_var = rate_change.rolling(window=window_size, min_periods=1).var()
            temporal_features[f'motion_var_w{window}_{col}'] = roll_var.fillna(0).replace([np.inf, -np.inf], 0)
            
            # Signal energy (accumulated signal strength)
            roll_energy = (df[col] ** 2).rolling(window=window_size, min_periods=1).mean()
            temporal_features[f'signal_energy_w{window}_{col}'] = roll_energy.fillna(0).replace([np.inf, -np.inf], 0)
    
    # === 5. Peak-to-Peak Range (Maximum Motion Amplitude) ===
    # Measures maximum range of signal variation in local window
    # High P2P: dynamic activities (walking, falling)
    # Low P2P: static activities (sitting, sleeping, standing)
    for col in numeric_cols[:5]:
        rate_abs = df[col].diff().abs()
        peak2peak = rate_abs.rolling(window=20, min_periods=1).max().fillna(0)
        temporal_features[f'peak2peak_{col}'] = peak2peak.replace([np.inf, -np.inf], 0)
    
    # === 6. Cross-Channel Motion Correlation ===
    # Mean rate of change across all packet features
    # Indicates overall system-level motion detection
    all_channels_motion = df[numeric_cols].diff().abs().mean(axis=1)
    temporal_features['cross_channel_motion_mean'] = all_channels_motion.fillna(0).replace([np.inf, -np.inf], 0)
    
    # === 7. Global Motion Statistics (Activity Signatures) ===
    # These aggregated features help distinguish activity classes
    
    # Overall rate of change magnitude
    diff_abs = df[numeric_cols].diff().abs()
    temporal_features['overall_motion_mean'] = diff_abs.mean(axis=1).fillna(0).replace([np.inf, -np.inf], 0)
    temporal_features['overall_motion_max'] = diff_abs.max(axis=1).fillna(0).replace([np.inf, -np.inf], 0)
    temporal_features['overall_motion_std'] = diff_abs.std(axis=1).fillna(0).replace([np.inf, -np.inf], 0)
    
    # Overall acceleration magnitude
    diff2_abs = df[numeric_cols].diff().diff().abs()
    temporal_features['overall_accel_mean'] = diff2_abs.mean(axis=1).fillna(0).replace([np.inf, -np.inf], 0)
    temporal_features['overall_accel_max'] = diff2_abs.max(axis=1).fillna(0).replace([np.inf, -np.inf], 0)
    
    # Signal volatility (variation across channels)
    # High: inconsistent signal (movement)
    # Low: stable signal (static)
    signal_std = df[numeric_cols].std(axis=1)
    temporal_features['signal_volatility'] = signal_std.fillna(0).replace([np.inf, -np.inf], 0)
    
    # === 8. Zero-Crossing Rate (Periodicity Detection) ===
    # Counts how often the rate of change switches sign
    # High ZCR: periodic motion (walking - alternating steps)
    # Low ZCR: monotonic or static motion (sitting, sleeping)
    rate_change_all = df[numeric_cols].diff()
    zcr = ((rate_change_all[:-1].values * rate_change_all[1:].values < 0).sum(axis=1) / len(numeric_cols))
    temporal_features['motion_zero_crossing_rate'] = pd.Series(zcr, index=df.index[:-1]).reindex(df.index, fill_value=0)
    
    print(f"Created {len(temporal_features.columns)} temporal features")
    
    # === Quality Control: Verify No Invalid Values ===
    nan_count = temporal_features.isna().sum().sum()
    inf_count = np.isinf(temporal_features.select_dtypes(include=[np.number])).sum().sum()
    
    if nan_count > 0:
        print(f"⚠️  Warning: {nan_count} NaN values found, filling with 0")
        temporal_features = temporal_features.fillna(0)
    
    if inf_count > 0:
        print(f"⚠️  Warning: {inf_count} Inf values found, replacing with 0")
        temporal_features = temporal_features.replace([np.inf, -np.inf], 0)
    
    # Combine original features with temporal features
    df_enhanced = pd.concat([df, temporal_features], axis=1)
    
    print(f"✓ Final dataset shape: {df_enhanced.shape}")
    print(f"✓ Final NaN count: {df_enhanced.isna().sum().sum()}")
    
    return df_enhanced

# Extract temporal features for BOTH datasets
print("=== Extracting Temporal Features for Training Data ===")
df_train = extract_temporal_features(df_train)
print(f"Enhanced training shape: {df_train.shape}\n")

print("=== Extracting Temporal Features for Validation Data ===")
df_valid = extract_temporal_features(df_valid)
print(f"Enhanced validation shape: {df_valid.shape}\n")

# Build preprocessing pipeline
TARGET_COLUMN = "label"
FEATURE_COLUMNS = [col for col in df_train.columns if col != TARGET_COLUMN]

print(f"Total features for training: {len(FEATURE_COLUMNS)}")

# Label Encoder
le = ColumnTransformer(
    transformers=[
        ("encoder", OrdinalEncoder(), [TARGET_COLUMN])
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

# Feature Scaler (Z-score normalization)
scaler = ColumnTransformer(
    transformers=[
        ("scaler", StandardScaler(), FEATURE_COLUMNS)
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

# PCA for dimensionality reduction (preserve 95% variance)
pca = ColumnTransformer(
    transformers=[
        ("pca", PCA(n_components=0.95), FEATURE_COLUMNS)
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

# Combine into pipeline
pipeline = make_pipeline(le, scaler, pca)
pipeline.set_output(transform="pandas")

# FIT pipeline on training data, TRANSFORM both train and validation
print("=== Applying Preprocessing Pipeline ===")
df_train = pipeline.fit_transform(df_train)
df_valid = pipeline.transform(df_valid)

print(f"Final training shape after PCA: {df_train.shape}")
print(f"Final validation shape after PCA: {df_valid.shape}")
print(f"Dimensionality reduction: {len(FEATURE_COLUMNS)} → {df_train.shape[1] - 1} features (95% variance retained)")

In [ ]:
# Use separate train and validation datasets (NO train_test_split!)
X_train = df_train.drop(TARGET_COLUMN, axis=1).to_numpy()
y_train = df_train[TARGET_COLUMN].to_numpy()

X_valid = df_valid.drop(TARGET_COLUMN, axis=1).to_numpy()
y_valid = df_valid[TARGET_COLUMN].to_numpy()

print(f"Training samples: {X_train.shape[0]}")
print(f"Validation samples: {X_valid.shape[0]}")

In [ ]:
BATCH_SIZE = 256


X_train = torch.tensor(X_train, dtype = torch.float32)
y_train = torch.tensor(y_train, dtype = torch.int64)

X_valid = torch.tensor(X_valid, dtype = torch.float32)
y_valid = torch.tensor(y_valid, dtype = torch.int64)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size = BATCH_SIZE, shuffle = True)
val_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size = BATCH_SIZE)


In [ ]:
# Auto-detect device: CUDA if available, otherwise CPU (macOS compatible)
if torch.cuda.is_available():
    device = "cuda"
    print(f"Using device: {device} (GPU)")
elif torch.backends.mps.is_available():
    device = "mps"
    print(f"Using device: {device} (Apple Silicon GPU)")
else:
    device = "cpu"
    print(f"Using device: {device} (CPU)")

In [ ]:
class SimpleMLP(nn.Module):
    """
    Regularized MLP with L2 regularization and strong dropout for better generalization.
    Designed to prevent overfitting on WiFi HAR task with velocity features.
    """
    def __init__(self, input_size, num_classes, dropout_rate=0.6):
        super(SimpleMLP, self).__init__()
        
        # Smaller architecture with aggressive regularization
        self.fc1 = nn.Linear(input_size, 96)  # Reduced further
        self.bn1 = nn.BatchNorm1d(96)
        self.dropout1 = nn.Dropout(dropout_rate)
        
        self.fc2 = nn.Linear(96, 48)  # Reduced
        self.bn2 = nn.BatchNorm1d(48)
        self.dropout2 = nn.Dropout(dropout_rate)
        
        self.fc3 = nn.Linear(48, 24)  # Tighter bottleneck
        self.bn3 = nn.BatchNorm1d(24)
        self.dropout3 = nn.Dropout(dropout_rate * 0.8)
        
        self.fc4 = nn.Linear(24, num_classes)
        
        # Initialize weights with smaller values to prevent overfitting
        self._initialize_weights()
        
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = torch.relu(x)
        x = self.dropout1(x)
        
        x = self.fc2(x)
        x = self.bn2(x)
        x = torch.relu(x)
        x = self.dropout2(x)
        
        x = self.fc3(x)
        x = self.bn3(x)
        x = torch.relu(x)
        x = self.dropout3(x)
        
        return self.fc4(x)

In [ ]:
def mixup_data(x, y, alpha=0.2):
    """
    Apply mixup data augmentation to reduce overfitting.
    Returns mixed inputs, pairs of targets, and lambda.
    """
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Mixup loss function."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

EPOCHS = 80  # More epochs with better early stopping
EARLY_STOPPING_PATIENCE = 20
MODEL_PATH = Path("modelcheckpoints/best_model.pt")
MIXUP_ALPHA = 0.2  # Mixup augmentation strength

patience = EARLY_STOPPING_PATIENCE
best_val_acc = 0.0
best_val_loss = float('inf')

# Use improved MLP model
model = SimpleMLP(input_size=X_train.shape[1], num_classes=len(le.named_transformers_["encoder"].categories_[0]), dropout_rate=0.6)
model = model.to(device)

# Compute class weights for imbalanced data
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train.numpy())
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

# Strong regularization
criterion = nn.CrossEntropyLoss(label_smoothing=0.2, weight=class_weights)  # Increased label smoothing
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)  # Stronger weight decay
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=7, verbose=True)

# Track metrics
train_losses, val_losses, val_accs = [], [], []

for epoch in range(EPOCHS):
    # === Training Phase with Mixup ===
    model.train()
    running_loss = 0.0
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    
    for xb, yb in progress:
        xb, yb = xb.to(device), yb.to(device)
        
        # Apply Mixup augmentation
        mixed_x, y_a, y_b, lam = mixup_data(xb, yb, alpha=MIXUP_ALPHA)
        
        optimizer.zero_grad()
        preds = model(mixed_x)
        loss = mixup_criterion(criterion, preds, y_a, y_b, lam)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    # === Validation Phase (NO Mixup) ===
    model.eval()
    val_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)
            val_loss += loss.item()
            correct += (preds.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    # Update learning rate based on validation LOSS
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    # Early stopping based on VALIDATION LOSS (more stable than accuracy)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)
        patience = EARLY_STOPPING_PATIENCE
        print(f"  ✓ Best model saved! (Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f})")
    else:
        patience -= 1
        if patience == 0:
            print(f"Early stopping at epoch {epoch+1}. Best Val Acc: {best_val_acc:.4f}")
            break

print(f"\n=== Training Complete ===")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")
print(f"Best Validation Loss: {best_val_loss:.4f}")

# Load best model
model.load_state_dict(torch.load(MODEL_PATH))
model.eval()
print("Loaded best model for evaluation.")

In [ ]:
model.eval()

torch.save(model.state_dict(), "modelcheckpoints/TrainTestSplit.pt")

In [ ]:
def plot_confusion_matrix(cm, categories, title="Confusion Matrix with Counts and Percentages"):
    """
    Plot confusion matrix with both counts and normalized percentages in each cell.
    
    cm: numpy array (confusion matrix, counts)
    categories: list of class names
    """
    cm = np.array(cm, dtype=int)  # ensure integer counts
    cm_normalized = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]

    # Build annotation labels with both count and percentage
    labels = np.array([
        [f"{count}\n({perc:.2%})" if cm[i, j] != 0 else ""
         for j, (count, perc) in enumerate(zip(row, row_norm))]
        for i, (row, row_norm) in enumerate(zip(cm, cm_normalized))
    ])

    plt.figure(figsize=(9,7))
    ax = sns.heatmap(cm, annot=labels, fmt="", cmap="Blues", cbar=True,
                     xticklabels=categories, yticklabels=categories,
                     annot_kws={"size":10, "weight":"bold"})

    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    plt.tight_layout()
    plt.show()

In [ ]:
model.to(device)

categories = le.named_transformers_['encoder'].categories_[0]

cm = np.zeros((5,5))

for X_batch, y_batch in tqdm(val_loader):
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    pred = model(X_batch).argmax(dim = 1)
    pred = pred.to('cpu')
    y_batch = y_batch.to('cpu')
    cm += confusion_matrix(y_batch, pred, labels = np.arange(len(categories)))

cm


In [ ]:
# Confusion Matrix on TRAINING data
model.to(device)

categories = le.named_transformers_['encoder'].categories_[0]

cm_train = np.zeros((5,5), dtype=int)

for X_batch, y_batch in tqdm(train_loader, desc="Computing training confusion matrix"):
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    pred = model(X_batch).argmax(dim=1).cpu()
    y_batch = y_batch.cpu()
    cm_train += confusion_matrix(y_batch, pred, labels=np.arange(len(categories)))

# Plot it
plot_confusion_matrix(cm_train, categories, title="Confusion Matrix on Training Data")

In [ ]:
cm
cm_df

In [ ]:
# Confusion Matrix on VALIDATION data
model.to(device)

categories = le.named_transformers_['encoder'].categories_[0]

cm_valid = np.zeros((5,5), dtype=int)

for X_batch, y_batch in tqdm(val_loader, desc="Computing validation confusion matrix"):
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    pred = model(X_batch).argmax(dim=1).cpu()
    y_batch = y_batch.cpu()
    cm_valid += confusion_matrix(y_batch, pred, labels=np.arange(len(categories)))

# Plot it
plot_confusion_matrix(cm_valid, categories, title="Confusion Matrix on Validation Data")